In [1]:
# analyze_data.py

import sqlite3
from collections import defaultdict
from pathlib import Path


def get_db_path() -> str:
    """Return path to the SQLite database."""
    try:
        base_dir = Path(__file__).parent
    except NameError:
        # Fallback for Jupyter
        base_dir = Path.cwd()
    return str(base_dir / "data" / "myproject.db")


# ========== QUERY 1 ==========

def query1_customers_by_city(db_path: str, city: str):
    """Find all customers from a specific city (SQL only).

    Business question:
        Which customers are in a given city for local marketing?
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        results = cursor.execute(
            """
            SELECT customer_id, name, email, city, join_date
            FROM customers
            WHERE city = ?;
            """,
            (city,),
        ).fetchall()
    return results


# ========== QUERY 2 ==========

def query2_revenue_by_category_sql(db_path: str):
    """Calculate total revenue by product category (SQL only).

    Business question:
        Which product categories generate the most revenue?

    Revenue is defined as:
        SUM(quantity * price)
    """
    sql = """
        SELECT p.category,
               SUM(o.quantity * p.price) AS total_revenue
        FROM orders AS o
        JOIN products AS p
            ON o.product_id = p.product_id
        GROUP BY p.category
        ORDER BY total_revenue DESC;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        results = cursor.execute(sql).fetchall()
    return results


# ========== QUERY 3 ==========

def query3_top_customers_sql(db_path: str, limit: int = 5):
    """Find top N customers by total spending (SQL version).

    Business question:
        Who are our most valuable customers?

    Total spending = SUM(quantity * price) over all their orders.
    """
    sql = """
        SELECT c.customer_id,
               c.name,
               SUM(o.quantity * p.price) AS total_spent
        FROM orders AS o
        JOIN customers AS c
            ON o.customer_id = c.customer_id
        JOIN products AS p
            ON o.product_id = p.product_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        results = cursor.execute(sql, (limit,)).fetchall()
    return results


def query3_top_customers_python(db_path: str, limit: int = 5):
    """Find top N customers by total spending (Python version).

    Uses a simple JOIN query to get detailed rows, then:
        - loops in Python,
        - aggregates per customer with a dictionary,
        - sorts to get the top N.
    """
    sql = """
        SELECT c.customer_id,
               c.name,
               o.quantity,
               p.price
        FROM orders AS o
        JOIN customers AS c
            ON o.customer_id = c.customer_id
        JOIN products AS p
            ON o.product_id = p.product_id;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        rows = cursor.execute(sql).fetchall()

    # rows: (customer_id, name, quantity, price)
    totals = defaultdict(float)
    names = {}

    for customer_id, name, quantity, price in rows:
        names[customer_id] = name
        totals[customer_id] += quantity * price

    aggregated = [
        (cust_id, names[cust_id], total_spent)
        for cust_id, total_spent in totals.items()
    ]
    aggregated.sort(key=lambda x: x[2], reverse=True)

    return aggregated[:limit]


# ========== QUERY 4 ==========

def query4_city_sales_sql(db_path: str):
    """Summarize revenue and number of orders by customer city (SQL version).

    Business question:
        Which customer locations generate the most sales?

    For each city, we compute:
        - total revenue = SUM(quantity * price)
        - number of orders  = COUNT(DISTINCT order_id)
    """
    sql = """
        SELECT c.city,
               SUM(o.quantity * p.price) AS total_revenue,
               COUNT(DISTINCT o.order_id) AS num_orders
        FROM orders AS o
        JOIN customers AS c
            ON o.customer_id = c.customer_id
        JOIN products AS p
            ON o.product_id = p.product_id
        GROUP BY c.city
        ORDER BY total_revenue DESC;
    """

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        results = cursor.execute(sql).fetchall()
    return results


def query4_city_sales_python(db_path: str):
    """Summarize revenue and number of orders by customer city (Python version)."""
    sql = """
        SELECT o.order_id,
               c.city,
               o.quantity,
               p.price
        FROM orders AS o
        JOIN customers AS c
            ON o.customer_id = c.customer_id
        JOIN products AS p
            ON o.product_id = p.product_id;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        rows = cursor.execute(sql).fetchall()

    # rows: (order_id, city, quantity, price)
    revenue_by_city = defaultdict(float)
    orders_by_city = defaultdict(set)

    for order_id, city, quantity, price in rows:
        revenue_by_city[city] += quantity * price
        orders_by_city[city].add(order_id)

    results = []
    for city in revenue_by_city:
        total_rev = revenue_by_city[city]
        num_orders = len(orders_by_city[city])
        results.append((city, total_rev, num_orders))

    # Sort by total revenue desc
    results.sort(key=lambda x: x[1], reverse=True)
    return results


# ========== QUERY 5 ==========

def query5_product_profit_sql(db_path: str, limit: int = 10):
    """Calculate product-level profitability (SQL version).

    Business question:
        Which individual products generate the highest profit?

    For each product:
        revenue = SUM(quantity * price)
        cost    = SUM(quantity * cost)
        profit  = revenue - cost
    """
    sql = """
        SELECT p.product_id,
               p.name,
               p.category,
               SUM(o.quantity * p.price) AS revenue,
               SUM(o.quantity * p.cost)  AS total_cost,
               SUM(o.quantity * (p.price - p.cost)) AS profit
        FROM orders AS o
        JOIN products AS p
            ON o.product_id = p.product_id
        GROUP BY p.product_id, p.name, p.category
        ORDER BY profit DESC
        LIMIT ?;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        results = cursor.execute(sql, (limit,)).fetchall()
    return results


def query5_product_profit_python(db_path: str, limit: int = 10):
    """Calculate product-level profitability (Python version)."""
    sql = """
        SELECT p.product_id,
               p.name,
               p.category,
               o.quantity,
               p.price,
               p.cost
        FROM orders AS o
        JOIN products AS p
            ON o.product_id = p.product_id;
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        rows = cursor.execute(sql).fetchall()

    # rows: (product_id, name, category, quantity, price, cost)
    totals = defaultdict(lambda: {"name": "", "category": "", "revenue": 0.0, "cost": 0.0})

    for product_id, name, category, quantity, price, cost in rows:
        record = totals[product_id]
        record["name"] = name
        record["category"] = category
        record["revenue"] += quantity * price
        record["cost"] += quantity * cost

    results = []
    for product_id, stats in totals.items():
        profit = stats["revenue"] - stats["cost"]
        results.append(
            (product_id, stats["name"], stats["category"], stats["revenue"], stats["cost"], profit)
        )

    # sort by profit desc
    results.sort(key=lambda x: x[5], reverse=True)
    return results[:limit]


# ========== MAIN ==========

def main():
    db_path = get_db_path()
    print("=== Analysis Results ===\n")

    # ---- Query 1 ----
    print("1. Customers in Boston:")
    for row in query1_customers_by_city(db_path, "Boston"):
        print(row)
    print()

    # ---- Query 2 ----
    print("2. Revenue by product category (SQL):")
    for row in query2_revenue_by_category_sql(db_path):
        print(row)
    print()

    # ---- Query 3 ----
    print("3. Top customers by total spending (SQL):")
    for row in query3_top_customers_sql(db_path, limit=5):
        print(row)
    print("\n3b. Top customers by total spending (Python):")
    for row in query3_top_customers_python(db_path, limit=5):
        print(row)
    print()

    # ---- Query 4 ----
    print("4. City-level sales summary (SQL):")
    for row in query4_city_sales_sql(db_path):
        print(row)
    print("\n4b. City-level sales summary (Python):")
    for row in query4_city_sales_python(db_path):
        print(row)
    print()

    # ---- Query 5 ----
    print("5. Product profitability (SQL):")
    for row in query5_product_profit_sql(db_path, limit=10):
        print(row)
    print("\n5b. Product profitability (Python):")
    for row in query5_product_profit_python(db_path, limit=10):
        print(row)
    print()


if __name__ == "__main__":
    main()




=== Analysis Results ===

1. Customers in Boston:
(1, 'Sarah Johnson', 'sarah.j@email.com', 'Boston', '2023-01-15')
(3, 'Emma Davis', 'emma.d@email.com', 'Boston', '2023-03-10')
(5, 'Olivia Brown', 'olivia.b@email.com', 'Boston', '2023-04-05')
(8, 'Benjamin Lee', 'ben.lee@email.com', 'Boston', '2023-05-15')
(11, 'Isabella Thomas', 'i.thomas@email.com', 'Boston', '2023-06-28')
(14, 'Daniel White', 'd.white@email.com', 'Boston', '2023-08-05')
(17, 'Amelia Lewis', 'amelia.l@email.com', 'Boston', '2023-09-15')
(20, 'David Allen', 'd.allen@email.com', 'Boston', '2023-10-25')
(22, 'Jackson King', 'jack.k@email.com', 'Boston', '2023-11-20')
(25, 'Emily Scott', 'emily.s@email.com', 'Boston', '2024-02-05')
(27, 'Sofia Adams', 'sofia.a@email.com', 'Boston', '2024-04-01')
(30, 'Joshua Carter', 'josh.c@email.com', 'Boston', '2024-07-15')

2. Revenue by product category (SQL):
('Electronics', 3506.06)
('Accessories', 748.67)
('Office Supplies', 396.65999999999997)

3. Top customers by total spendin